# 04 · code_paths — Custom Python 모듈 번들링 (그리고 그 제약)

## `code_paths` 란

`mlflow.pyfunc.log_model(code_paths=[...])` 에 전달된 로컬 `.py` 파일 / 디렉토리 / `.whl` 은
모델 디렉토리의 `code/` 하위로 복사되고, **load 시 `sys.path` 앞에 추가**됩니다.

## 동작 흐름

```
log_model(code_paths=["utils.py", "featurizers/"])
  → model_uri/code/utils.py
  → model_uri/code/featurizers/...

load_model()
  → sys.path.insert(0, "<model_uri>/code")
  → `from utils import clean` 가능
```

## 함정들

1. **평탄화 (flattening)** — `src/utils.py` 를 주면 `code/utils.py` 로 들어감 (`code/src/utils.py` 아님)
2. **중첩 패키지** — `from mypkg.sub import foo` 같은 import 는 패키지 디렉토리 전체를 통째로 줘야 함
3. **Relative import** — `from .x import y` 는 종종 깨짐
4. **Transitive deps** — pyproject.toml 처럼 dep 캡처 안 됨. `pip_requirements` 에 직접 넣어야 함

> 결론: 트리비얼한 한두 파일은 `code_paths` OK, 그 이상은 **wheel 빌드 권장** → `05_uv_wheel`

In [ ]:
%run ./config

In [ ]:
import mlflow, os, shutil, pandas as pd
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

## Step 1. 로컬 작업 디렉토리에 custom 모듈 작성

In [ ]:
work_dir = "/tmp/codepaths_demo"
shutil.rmtree(work_dir, ignore_errors=True)
os.makedirs(work_dir, exist_ok=True)

# 모듈 1: 단일 파일 — preprocessing.py
preprocessing_code = '''
"""고객 데이터 전처리 — code_paths 데모용."""
import pandas as pd

def add_lifetime_value(df: pd.DataFrame) -> pd.DataFrame:
    """LTV proxy = monthly * tenure."""
    df = df.copy()
    df["ltv"] = df["monthly_charges"] * df["tenure_months"]
    return df

def risk_bucket(tickets: int) -> str:
    if tickets >= 10: return "high"
    if tickets >= 4:  return "medium"
    return "low"
'''
with open(f"{work_dir}/preprocessing.py", "w") as f:
    f.write(preprocessing_code)

# 모듈 2: 패키지 — featurizers/
os.makedirs(f"{work_dir}/featurizers", exist_ok=True)
with open(f"{work_dir}/featurizers/__init__.py", "w") as f:
    f.write("from .text import normalize  # noqa: F401\n")
with open(f"{work_dir}/featurizers/text.py", "w") as f:
    f.write('def normalize(s: str) -> str:\n    return s.lower().strip()\n')

# 디렉토리 트리 확인
for root, dirs, files in os.walk(work_dir):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), work_dir))

## Step 2. PythonModel — code_paths 에서 import

In [ ]:
import sys
sys.path.insert(0, work_dir)   # 노트북에서도 import 되도록

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from preprocessing import add_lifetime_value   # ← work_dir 의 모듈

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()
pdf = add_lifetime_value(pdf)

FEATURES = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets", "ltv"]
X_train, X_test, y_train, y_test = train_test_split(
    pdf[FEATURES], pdf["churned"], test_size=0.2, random_state=42
)
rf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42).fit(X_train, y_train)


class ChurnWithPreproc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # code/ 가 sys.path 에 자동 추가되므로 그냥 import
        from preprocessing import add_lifetime_value
        from featurizers import normalize
        self.add_ltv   = add_lifetime_value
        self.normalize = normalize
        import joblib
        self.model = joblib.load(context.artifacts["model"])

    def predict(self, context, model_input: pd.DataFrame, params=None):
        # raw input 에 LTV 컬럼이 없을 수도 있음 — 전처리에서 추가
        if "ltv" not in model_input.columns:
            model_input = self.add_ltv(model_input)
        return pd.DataFrame({"prediction": self.model.predict(model_input[FEATURES])})

## Step 3. RF 모델을 Volume 에 저장 + log_model

In [ ]:
import joblib
artifacts_dir = f"{volume_path}/04_codepaths"
os.makedirs(artifacts_dir, exist_ok=True)
model_path = f"{artifacts_dir}/rf.joblib"
joblib.dump(rf, model_path)

from mlflow.models import infer_signature

# input_example 은 LTV **없는** raw 데이터 — predict 가 자동으로 추가하는지 검증
raw_example = pdf[["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]].iloc[:5]
sig = infer_signature(raw_example, pd.DataFrame({"prediction": [0]*5}))

with mlflow.start_run(run_name="codepaths_demo"):
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=ChurnWithPreproc(),
        artifacts={"model": model_path},
        code_paths=[
            f"{work_dir}/preprocessing.py",   # 단일 파일
            f"{work_dir}/featurizers",        # 디렉토리 (패키지)
        ],
        signature=sig,
        input_example=raw_example,
        registered_model_name=model_codepaths,
        pip_requirements=[
            "scikit-learn==1.4.2",
            "joblib==1.4.0",
            "pandas==2.2.2",
        ],
    )

print(f"✓ {model_codepaths} v{info.registered_model_version}")

## Step 4. 모델 디렉토리 구조 확인 — code/ 안에 평탄화된 모습

In [ ]:
local = mlflow.artifacts.download_artifacts(info.model_uri)
for root, dirs, files in os.walk(local):
    level = root.replace(local, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

`code/` 안에 `preprocessing.py` 와 `featurizers/` 가 그대로 들어갔습니다.
만약 `code_paths=["src/preprocessing.py"]` 처럼 하위 경로를 줬다면 `code/preprocessing.py`
로 **평탄화** 되었을 겁니다 — 이게 nested package 가 깨지는 이유.

## Step 5. Load + 예측 — raw input 으로

In [ ]:
loaded = mlflow.pyfunc.load_model(info.model_uri)
display(loaded.predict(raw_example))

## Step 6. (시연) 평탄화 함정 — `src/` 안에 두면 어떻게 되는지

In [ ]:
# nested 경로 시연용
nested_dir = "/tmp/codepaths_nested"
shutil.rmtree(nested_dir, ignore_errors=True)
os.makedirs(f"{nested_dir}/src", exist_ok=True)
with open(f"{nested_dir}/src/helper.py", "w") as f:
    f.write('def hello():\n    return "from src/helper"\n')

with mlflow.start_run(run_name="flatten_demo"):
    info_flat = mlflow.pyfunc.log_model(
        name="model",
        python_model=ChurnWithPreproc(),
        artifacts={"model": model_path},
        code_paths=[f"{nested_dir}/src/helper.py"],   # ← src/ 하위
        pip_requirements=["scikit-learn==1.4.2", "joblib==1.4.0", "pandas==2.2.2"],
    )

flat_local = mlflow.artifacts.download_artifacts(info_flat.model_uri)
print("\ncode/ 안 파일 위치:")
for f in os.listdir(os.path.join(flat_local, "code")):
    print(f"  {f}")
print("\n→ src/ 디렉토리 구조는 사라지고 helper.py 만 code/ 에 평탄화됨")

## 정리

| 케이스 | 권장 |
| --- | --- |
| 단일 .py 파일 1-2개 | `code_paths` |
| 평탄한 단일 패키지 디렉토리 | `code_paths=["mypkg/"]` |
| 중첩 패키지 / relative import | **wheel 빌드** → `05_uv_wheel` |
| C-extension 의존 패키지 | **wheel 빌드** (manylinux 빌드) |
| Transitive deps 자동 관리 필요 | **wheel + pyproject.toml** |

→ 다음: **`05_uv_wheel`** — uv 로 wheel 빌드해서 best practice 로 패키징